# Lekcija 12 - Smanjenje Povijesti Razgovora s Agent Scratchpad-om

Ovaj bilježnik demonstrira kako upravljati kontekstom u dugim razgovorima koristeći Microsoft Agent Framework. Kako razgovori rastu, broj tokena se povećava — na kraju prelazeći prozor konteksta modela. Rješavamo to pomoću **uzorka za sažimanje konteksta** i **agent scratchpad-a** za trajnu memoriju.

## Što ćete naučiti:
1. **Zašto je upravljanje kontekstom važno**: Razumijevanje ograničenja tokena i prozora konteksta
2. **Agenti svjesni konteksta**: Izrada agenata koji upravljaju vlastitim kontekstom razgovora
3. **Uzorak sažimanja konteksta**: Korištenje alata za kondenzaciju povijesti razgovora
4. **Agent Scratchpad**: Trajna memorija koja preživljava smanjenje konteksta

## Preduvjeti:
- Azure OpenAI postavljanje s konfiguriranim varijablama okoline
- Razumijevanje osnovnih pojmova agenata iz prethodnih lekcija


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Zašto upravljanje kontekstom ima važnost

Svaki LLM ima ograničeni **prozor konteksta** — maksimalni broj tokena koje može obraditi u jednom zahtjevu. Kako se razgovor u više koraka odvija:

- **Broj tokena linearno raste** s svakom korisničkom porukom i odgovorom asistenta.
- **Tokeni upita dominiraju troškovima** jer se cijela povijest ponovno šalje svaki put.
- Na kraju razgovor **prelazi prozor konteksta** i model ili skraćuje ili daje pogrešku.

### Strategije upravljanja kontekstom

| Strategija | Kako funkcionira | Kompromis |
|---|---|---|
| **Skraćivanje** | Odbacivanje najstarijih poruka | Gubi se rani kontekst |
| **Sažimanje** | Sažimanje starijih poruka u rezime | Gubi se dio detalja, ali ključne točke ostaju |
| **Notes / Vanjska memorija** | Pohrana ključnih činjenica izvan razgovora | Zahtijeva pozive alatu, ali preživljava svako skraćivanje |

U ovom bilježniku kombiniramo **sažimanje** s **notes alatom** kako bi agent mogao održavati kontinuitet čak i kad je povijest razgovora sažeta.


## Kreiranje agenta svjesnog konteksta


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Simulacija dugog razgovora

Prođimo kroz višekratni razgovor kako bismo vidjeli kako se kontekst akumulira. Agent bi trebao zadržati ključne detalje (preferencije, proračun, datume putovanja) tijekom razgovora i pokazati kontinuitet.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Primijetite kako agent zadržava kontekst iz ranijih okreta — zna za Japan, sushi, hramove, fotografiju, budžet od 3000 dolara, solo putovanje i vremenski okvir u travnju. U kratkom razgovoru ovo dobro funkcionira, ali kako razgovor raste, cijela povijest postaje skupa za ponovno slanje.

Nastavimo razgovor s više okreta kako bismo vidjeli nakupljanje konteksta:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Obrasci sažimanja konteksta

Kako razgovor napreduje, možemo koristiti **alat za sažimanje** za sažimanje prikupljenog konteksta u kompaktan format. Agent poziva ovaj alat kako bi zabilježio ključne preferencije tako da, čak i ako se starije poruke izbrišu, bitne informacije budu sačuvane.

Ovaj obrazac je temelj za složenije smanjenje povijesti:
1. Agent prepoznaje ključne činjenice iz razgovora
2. Poziva alat za sažimanje da ih sačuva
3. Starije poruke se mogu sigurno ukloniti jer sažetak bilježi ono što je važno

U nastavku definiramo alat `summarize_preferences` koji agent može pozvati za bilježenje sažetka onoga što je naučio.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Sažetak

U ovoj lekciji naučili ste kako upravljati kontekstom u dugotrajnim razgovorima agenata koristeći Microsoft Agent Framework:

### Ključni Koncepti
- **Kontekstni prozori su ograničeni** — svaki token u povijesti razgovora košta i uračunava se u limit.
- **Alati za sažimanje** omogućuju agentu da sažme prikupljeni kontekst u kompaktne sažetke, smanjujući korištenje tokena uz očuvanje bitnih informacija.
- **Bilježnice agenata** pružaju trajnu vanjsku memoriju koja preživljava smanjenje bilo kojeg razgovora.

### Što ste izgradili
- **Agent svjestan konteksta** koji održava kontinuitet kroz višestruke krugove razgovora
- **Alat za sažimanje** (`summarize_preferences`) koji bilježi ključne korisničke detalje u kompaktni format
- **Razgovor s više krugova** koji demonstrira zadržavanje konteksta i upravljanje promjenama

### Primjene u stvarnom svijetu
- **Botovi za korisničku službu**: Pamte preferencije kroz duge sesije podrške
- **Osobni asistenti**: Prate tekuće projekte bez ponovnog objašnjavanja konteksta
- **Obrazovni tutori**: Održavaju napredak učenika kroz mnoge interakcije

### Sljedeći koraci
- Implementirati puni alat bilježnice s trajnijim pohranjivanjem u datoteke
- Dodati automatsko skraćivanje povijesti nakon sažimanja
- Kombinirati s vektorskim bazama podataka za semantičko pretraživanje memorije
- Izgraditi agente koji mogu nastaviti razgovore nakon nekoliko dana s punim kontekstom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
